# Pricing and hedging a variable annuity maturity guarantee

This report prices a guaranteed minimum maturity benefit (GMMB) by Monte Carlo,
checks the price against the closed-form Black-Scholes value, and studies
how well the guarantee can be delta-hedged: as a function of rebalancing
frequency and transaction costs, when the hedging model is wrong (the market
follows Heston), and on historical S&P 500 paths.

Every number below is produced by the `ghedge` package from fixed random seeds,
so re-running the notebook reproduces it.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "ghedge").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

from ghedge import GBM, GMMBContract, Heston, Decrements
from ghedge import blackscholes as bs, heston
from ghedge.montecarlo import METHODS, price_gmmb, convergence_study, convergence_slope
from ghedge.greeks import pathwise_delta, fd_delta, pathwise_vega, fd_vega, fd_gamma
from ghedge.hedging import BSHedger, HestonHedger, backtest, rebalancing_table, risk_summary
from ghedge.data import load_index
from ghedge.historical import historical_backtest

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True, "grid.alpha": 0.3})
%matplotlib inline

## 1. Contract and assumptions

* Single premium of 100 invested in a fund that tracks an equity index less a
  continuous management fee $m$: $F_t = S_t e^{-mt}$.
* At maturity $T$ each surviving policy receives $\max(F_T, G)$; the insurer
  owes the shortfall $(G - F_T)^+$, a put on the fund.
* Policyholders leave through lapses (constant force) and death
  (Gompertz-Makeham mortality). Decrements are deterministic and independent
  of the market, so the cohort liability is ${}_Tp_x \cdot \text{Put}$.
* Flat risk-free rate $r$. Under Black-Scholes the fund is a stock with
  dividend yield $m$:

$$V_0 = {}_Tp_x\left(G e^{-rT} N(-d_2) - F_0 e^{-mT} N(-d_1)\right),\quad
d_{1,2} = \frac{\ln(F_0/G) + (r - m \pm \sigma^2/2)T}{\sigma\sqrt{T}}.$$

In [ ]:
contract = GMMBContract(premium=100.0, guarantee=100.0, maturity=10.0, fee=0.02, rate=0.03,
                        decrements=Decrements(age=55, lapse=0.03))
sigma = 0.2
display(pd.Series({"premium": contract.premium, "guarantee": contract.guarantee,
                   "maturity (years)": contract.maturity, "fee": contract.fee, "rate": contract.rate,
                   "issue age": contract.decrements.age, "lapse force": contract.decrements.lapse,
                   "P(in force at maturity)": contract.maturity_survival,
                   "closed-form value": float(bs.gmmb_value(contract, sigma))}, name="value").to_frame())

t = np.linspace(0, 30, 301)
fig, ax = plt.subplots()
ax.plot(t, contract.decrements.survival(t), label="lapse + mortality")
ax.plot(t, Decrements(age=55, lapse=0.0).survival(t), label="mortality only")
ax.set_xlabel("years since issue"); ax.set_ylabel("fraction in force"); ax.legend()
ax.set_title("Cohort decrements (issue age 55)")
plt.show()

## 2. Monte Carlo price against the closed form

200,000 paths per cell, plain Monte Carlo and antithetic + control variates.
The z-score is (MC - closed form) / standard error; the test suite asserts
$|z| < 3$ on this grid.

In [ ]:
rows = []
for g in [80.0, 100.0, 120.0]:
    for T in [1.0, 5.0, 10.0, 20.0]:
        c = contract.with_(guarantee=g, maturity=T)
        exact = float(bs.gmmb_value(c, sigma))
        for method in ["plain", "antithetic_control"]:
            res = price_gmmb(c, GBM(sigma), 200_000, method, rng=int(g * 100 + T))
            lo, hi = res.ci()
            rows.append({"guarantee": g, "maturity": T, "method": method, "closed_form": exact,
                         "mc": res.estimate, "std_error": res.std_error,
                         "ci95_low": lo, "ci95_high": hi,
                         "z": (res.estimate - exact) / res.std_error})
grid = pd.DataFrame(rows)
display(grid)
print(f"max |z| over {len(grid)} estimates: {grid['z'].abs().max():.2f}")

## 3. Variance reduction

* **Antithetic variates** pair each normal draw $Z$ with $-Z$.
* **Control variate** (GBM): the discounted terminal index $e^{-rT}S_T$, whose
  mean $S_0$ is known. The coefficient is the least-squares $\beta$.
* **Control variates** (Heston): the discounted index plus the guarantee
  payoff on a *shadow* GBM path driven by the same Brownian motion with
  volatility $\sqrt{\bar v}$ (the expected average variance). Its value is
  known in closed form.

The factor is the variance per simulated path of plain Monte Carlo divided by
that of each method, i.e. the saving in paths for the same accuracy.

In [ ]:
def vr_table(model, c, n, **kw):
    res = {m: price_gmmb(c, model, n, m, rng=7, **kw) for m in METHODS}
    base = res["plain"].variance_per_path
    return pd.DataFrame({m: {"estimate": r.estimate, "std_error": r.std_error,
                             "variance_per_path": r.variance_per_path,
                             "variance_reduction": base / r.variance_per_path}
                         for m, r in res.items()}).T

print("GBM, 10-year at-the-money guarantee, 400,000 paths")
display(vr_table(GBM(sigma), contract, 400_000))

heston_model = Heston(v0=0.04, kappa=1.5, theta=0.04, xi=0.5, rho=-0.7)
c5 = contract.with_(maturity=5.0)
print("Heston, 5-year guarantee, 100,000 paths, 50 steps a year; "
      f"semi-analytic value {float(heston.gmmb_value_and_greeks(c5, heston_model)[0][0]):.4f}")
display(vr_table(heston_model, c5, 100_000, steps_per_year=50))

## 4. Convergence

RMSE of 60 independent estimates around the closed form for each path count.
Monte Carlo error should fall like $N^{-1/2}$: a slope of $-0.5$ on a log-log
plot.

In [ ]:
sizes = [500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000]
conv = {m: convergence_study(contract, GBM(sigma), sizes, n_reps=60, method=m, seed=3)
        for m in ["plain", "antithetic_control"]}
fig, ax = plt.subplots()
for m, df in conv.items():
    ax.loglog(df["n_paths"], df["rmse"], "o-", label=f"{m} (slope {convergence_slope(df):.3f})")
ref = conv["plain"]["rmse"].iloc[0] * np.sqrt(sizes[0] / np.array(sizes))
ax.loglog(sizes, ref, "k--", alpha=0.5, label=r"$N^{-1/2}$ reference")
ax.set_xlabel("paths N"); ax.set_ylabel("RMSE vs closed form"); ax.legend()
ax.set_title("Monte Carlo convergence")
plt.show()
display(pd.concat({m: df.set_index("n_paths")["rmse"] for m, df in conv.items()}, axis=1))

## 5. Greeks

Pathwise derivatives differentiate the payoff along each path:
$\partial (G - F_T)^+/\partial S_0 = -\mathbf 1\{F_T < G\}\, F_T / S_0$ and, under GBM,
$\partial F_T/\partial\sigma = F_T (W_T - \sigma T)$. Finite differences use
central bumps with common random numbers. Gamma has no pathwise estimator
(the payoff has a kink), so only the finite-difference value is shown.

In [ ]:
rows = []
for g, T in [(80.0, 5.0), (100.0, 10.0), (120.0, 2.0)]:
    c = contract.with_(guarantee=g, maturity=T)
    for name, pw, fd, exact in [
        ("delta", pathwise_delta(c, GBM(sigma), 200_000, seed=1), fd_delta(c, GBM(sigma), 200_000, seed=1),
         float(bs.gmmb_delta(c, sigma))),
        ("vega", pathwise_vega(c, GBM(sigma), 200_000, seed=2), fd_vega(c, GBM(sigma), 200_000, seed=2),
         float(bs.gmmb_vega(c, sigma))),
    ]:
        rows.append({"guarantee": g, "maturity": T, "greek": name, "closed_form": exact,
                     "pathwise": pw.estimate, "pathwise_se": pw.std_error,
                     "finite_diff": fd.estimate, "fd_se": fd.std_error})
    gm = fd_gamma(c, GBM(sigma), 400_000, seed=3)
    rows.append({"guarantee": g, "maturity": T, "greek": "gamma", "closed_form": float(bs.gmmb_gamma(c, sigma)),
                 "pathwise": np.nan, "pathwise_se": np.nan, "finite_diff": gm.estimate, "fd_se": gm.std_error})
display(pd.DataFrame(rows))

c2 = contract.with_(maturity=2.0)
pw = pathwise_delta(c2, heston_model, 50_000, seed=4, steps_per_year=100)
fd = fd_delta(c2, heston_model, 50_000, seed=4, steps_per_year=100)
print(f"Heston 2-year delta: semi-analytic {float(heston.gmmb_value_and_greeks(c2, heston_model)[1][0]):.5f}, "
      f"pathwise {pw.estimate:.5f} ± {pw.std_error:.5f}, finite difference {fd.estimate:.5f} ± {fd.std_error:.5f}")

## 6. Delta hedging under Black-Scholes

The insurer charges the Black-Scholes value, holds $\Delta = \partial V/\partial S$
units of the index and the rest in cash, and rebalances daily (every trading
day), weekly (5 days) or monthly (21 days). Paths follow GBM with a 7%
real-world drift and $\sigma = 20\%$; the contract is a 5-year at-the-money
guarantee. Hedge P&L is discounted to issue and quoted per 100 of premium.
CVaR is the mean loss in the worst 5% of paths.

In [ ]:
hedge_contract = contract.with_(maturity=5.0)
n_hedge_paths = 2_000
gbm_paths = GBM(sigma, mu=0.07).simulate(100.0, hedge_contract.rate, 5.0, 5 * 252, n_hedge_paths, rng=21)
bs_table = rebalancing_table(gbm_paths, hedge_contract, [BSHedger(hedge_contract, sigma)],
                             cost_rates=(0.0, 0.001, 0.002))
display(bs_table.drop(columns=["strategy"]))

fig, ax = plt.subplots()
for name, k in [("monthly", 21), ("weekly", 5), ("daily", 1)]:
    pnl = backtest(gbm_paths, hedge_contract, BSHedger(hedge_contract, sigma), k).pnl
    ax.hist(pnl, bins=60, alpha=0.5, density=True, label=f"{name} (sd {pnl.std():.2f})")
ax.set_xlabel("discounted hedge P&L per 100 premium"); ax.legend()
ax.set_title("Hedge P&L under GBM, no transaction costs")
plt.show()

nc = bs_table[bs_table.cost_rate == 0].set_index("frequency")["std_pnl"]
print(f"std of hedge error: monthly {nc['monthly']:.3f}, weekly {nc['weekly']:.3f}, daily {nc['daily']:.3f}; "
      f"monthly/daily ratio {nc['monthly'] / nc['daily']:.2f} (sqrt(21) = {np.sqrt(21):.2f})")

With no costs the standard deviation of hedge error falls roughly like the
square root of the rebalancing interval. Proportional costs pull the mean
P&L down, and more so for frequent rebalancing: the frequency choice trades
hedge error against cost.

## 7. Model risk: hedging a Heston market with Black-Scholes

The market now follows Heston with $v_0 = \theta = 0.04$ ($\sqrt\theta = 20\%$),
$\kappa = 1.5$, $\xi = 0.5$, $\rho = -0.7$ and a 7% drift. Four hedgers:

| strategy | price charged | hedge ratio |
|---|---|---|
| BS at implied vol | Heston value | BS delta at the volatility that reproduces the Heston value |
| BS at $\sqrt\theta$ | BS value at 20% | BS delta at 20% (the "natural" but wrong model) |
| Heston delta | Heston value | $\partial V/\partial S$ with the true model, observing $v_t$ |
| Heston min-variance | Heston value | $\partial V/\partial S + \rho\xi S^{-1}\partial V/\partial v$ |

The Heston hedgers know the true parameters and observe the current variance;
even so, a hedge in the index alone cannot remove variance risk. All four run on
the same 2,000 Heston paths; the GBM result from section 6 is the baseline.

In [ ]:
heston_market = Heston(v0=0.04, kappa=1.5, theta=0.04, xi=0.5, rho=-0.7, mu=0.07)
heston_paths = heston_market.simulate(100.0, hedge_contract.rate, 5.0, 5 * 252, n_hedge_paths, rng=22)
heston_value = float(heston.gmmb_value_and_greeks(hedge_contract, heston_market)[0][0])
implied = bs.implied_vol_put(heston_value / hedge_contract.maturity_survival, 100.0, hedge_contract.guarantee,
                             hedge_contract.maturity, hedge_contract.rate, hedge_contract.fee)
print(f"Heston value {heston_value:.4f}; BS value at 20% {float(bs.gmmb_value(hedge_contract, 0.2)):.4f}; "
      f"implied vol {implied:.2%}")

hedgers = [BSHedger(hedge_contract, implied, label="BS at implied vol"),
           BSHedger(hedge_contract, 0.2, label="BS at sqrt(theta)"),
           HestonHedger(hedge_contract, heston_market, label="Heston delta"),
           HestonHedger(hedge_contract, heston_market, minimum_variance=True, label="Heston min-variance")]
freqs = {"weekly": 5, "monthly": 21}
mr = rebalancing_table(heston_paths, hedge_contract, hedgers, frequencies=freqs)
base = rebalancing_table(gbm_paths, hedge_contract, [BSHedger(hedge_contract, sigma, label="GBM market, BS delta")],
                         frequencies=freqs)
model_risk = pd.concat([base, mr], ignore_index=True)
display(model_risk[["strategy", "frequency", "price", "mean_pnl", "std_pnl", "var_95", "cvar_95"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
weekly = {h.label: backtest(heston_paths, hedge_contract, h, 5) for h in hedgers}
gbm_weekly = backtest(gbm_paths, hedge_contract, BSHedger(hedge_contract, sigma), 5)
ax = axes[0]
ax.hist(gbm_weekly.pnl, bins=60, alpha=0.5, density=True, label="GBM market, BS delta")
for lab in ["BS at implied vol", "Heston min-variance"]:
    ax.hist(weekly[lab].pnl, bins=60, alpha=0.5, density=True, label=f"Heston market, {lab}")
ax.set_xlabel("discounted hedge P&L"); ax.legend(fontsize=8); ax.set_title("Weekly hedging")

realised_var = (np.diff(np.log(heston_paths.index), axis=1) ** 2).sum(axis=1) / hedge_contract.maturity
ax = axes[1]
ax.scatter(np.sqrt(realised_var), weekly["BS at implied vol"].pnl, s=4, alpha=0.4)
ax.axvline(implied, color="k", ls="--", lw=1, label="hedging vol")
ax.set_xlabel("realised volatility over the contract"); ax.set_ylabel("hedge P&L")
ax.set_title("BS hedge error is driven by realised vs assumed vol"); ax.legend()
plt.tight_layout(); plt.show()

corr = np.corrcoef(np.sqrt(realised_var), weekly["BS at implied vol"].pnl)[0, 1]
print(f"correlation of BS hedge P&L with realised vol: {corr:.2f}")

In [ ]:
w = model_risk[model_risk.frequency == "weekly"].set_index("strategy")
gbm_sd = w.loc["GBM market, BS delta", "std_pnl"]
print("Weekly hedge-error standard deviation relative to the GBM baseline:")
for s in ["BS at implied vol", "BS at sqrt(theta)", "Heston delta", "Heston min-variance"]:
    print(f"  {s:22s} sd {w.loc[s, 'std_pnl']:.3f}  ({w.loc[s, 'std_pnl'] / gbm_sd:.1f}x)  "
          f"CVaR95 {w.loc[s, 'cvar_95']:.3f}  mean {w.loc[s, 'mean_pnl']:+.3f}")

**Reading the model-risk result.**

* With the market in Heston, the Black-Scholes hedge error is several times
  larger than the same hedge in a Black-Scholes market. More frequent
  rebalancing helps very little, because the error comes from the gap between
  realised and assumed volatility (right panel), not from discreteness.
* At the money the Heston value is below the Black-Scholes value at
  $\sqrt\theta$ (the implied volatility printed above is under 20%), so the
  "BS at $\sqrt\theta$" hedger overcharges. That shows up as a positive mean
  P&L, not as a smaller spread.
* Hedging with the true model's delta alone does **not** remove the error:
  the variance risk stays unhedged. The minimum-variance delta, which uses the
  correlation between index and variance, gives the smallest spread. Its mean
  P&L is lower under the real-world measure because it holds a larger short
  index position and so gives up more of the equity risk premium.
* Removing the remaining error needs a second instrument that trades
  volatility (for example listed index options), which is outside the scope here.

## 8. Historical backtest on the S&P 500

Daily S&P 500 closing levels from FRED (`scripts/fetch_index.py` refreshes
the committed snapshot). One-year GMMB contracts are issued every 21 trading
days; each is priced and hedged with Black-Scholes at the trailing one-year
realised volatility known at issue. The S&P 500 series is a price index, so
dividends are ignored, and the rate is a flat 2%. Transaction costs are 10 bp.
Windows overlap, so the contracts are far from independent.

In [ ]:
levels = load_index()
print(f"{len(levels)} daily observations, {levels.index[0].date()} to {levels.index[-1].date()}")
hist_contract = contract.with_(maturity=1.0, rate=0.02)
per_window, hist_summary = historical_backtest(levels, hist_contract, cost_rate=0.001)
display(hist_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(levels.index, levels.values)
axes[0].set_title("S&P 500 level"); axes[0].set_yscale("log")
gap = per_window["realised_vol"] - per_window["issue_vol"]
axes[1].scatter(gap, per_window["pnl_weekly"], s=12)
axes[1].set_xlabel("realised minus issue volatility"); axes[1].set_ylabel("weekly hedge P&L")
axes[1].set_title("Historical hedge error vs volatility surprise")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots()
for name in ["daily", "weekly", "monthly"]:
    ax.plot(per_window["issue_date"], per_window[f"pnl_{name}"], ".-", lw=0.8, label=name)
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("discounted hedge P&L per 100"); ax.set_title("Hedge P&L by issue date"); ax.legend()
plt.show()
print(f"correlation of weekly P&L with the volatility surprise: {np.corrcoef(gap, per_window['pnl_weekly'])[0, 1]:.2f}")

On real data the hedge error is dominated by volatility surprises (contracts
issued just before the 2020 sell-off lose the most), so rebalancing more often
does not reduce it the way it does in the simulated Black-Scholes world. In
this sample daily rebalancing even has the larger spread; costs explain part
of the gap in the mean, and one possible reason for the rest is that daily
returns were more volatile than returns measured over longer intervals, which
a monthly hedger never sees. With 96 overlapping windows this is weak evidence
either way. That
matches the Heston experiment in section 7: once volatility is stochastic,
rebalancing frequency is a second-order lever compared with the volatility
assumption.

## 9. Summary

* Monte Carlo prices agree with the closed form within three standard errors
  on a grid of guarantee levels and maturities; error falls like $N^{-1/2}$.
* Antithetic and control variates together cut the variance per path by about
  an order of magnitude for the 10-year guarantee.
* Pathwise and finite-difference Greeks agree with each other and with the
  closed form.
* Under Black-Scholes the hedge-error spread falls roughly like the square
  root of the rebalancing interval; costs grow as rebalancing gets more frequent.
* Under Heston, delta hedging with either model leaves a much larger error;
  the minimum-variance delta reduces but does not remove it.